# Разминка к ДЗ №1: кирпичики ML-пайплайна

Это **не** мини-версия ДЗ №1, а отработка отдельных «кирпичиков», из которых он собран —
каждый с автопроверкой. Освоив их здесь, в ДЗ №1 вы соберёте пайплайн на своих данных.

**Датасет фиксированный и офлайн** — `sklearn.load_breast_cancer` (бинарная классификация),
в него детерминированно добавлены категориальный признак и пропуски.

**Как работать.** Для каждого кирпичика три ячейки:
1. **задание** (markdown);
2. **реализация** — заполните тело функции вместо `# TODO`;
3. **проверка** — запустите: она напечатает `PASS/FAIL` по каждому критерию и сверит результат с эталоном.

Кирпичики: стратифицированный сплит · импутация без утечки · One-Hot · метрики руками ·
ROC-AUC при дисбалансе · кросс-валидация · GridSearchCV · важность признаков.

Сначала выполните две служебные ячейки ниже.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             accuracy_score, roc_auc_score)

SEED = 150

_data = load_breast_cancer(as_frame=True)
NUM_COLS = ["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness"]
df = _data.frame[NUM_COLS + ["target"]].copy()
df["area_cat"] = pd.qcut(df["mean area"], q=3, labels=["small", "medium", "large"]).astype(object)
df.loc[np.arange(0, len(df), 13), "mean smoothness"] = np.nan
CAT_COLS = ["area_cat"]

y = df["target"]
X = df.drop(columns="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)

Xn = X[NUM_COLS].copy()
Xn = Xn.fillna(Xn.median())

C_GRID = [0.01, 0.1, 1, 10]
PARAM_GRID = {"logisticregression__C": C_GRID}
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
def make_estimator():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,target,area_cat
0,17.99,10.38,122.80,1001.0,NaN,0,large
1,20.57,17.77,132.90,1326.0,0.08474,0,large
2,19.69,21.25,130.00,1203.0,0.10960,0,large
3,11.42,20.38,77.58,386.1,0.14250,0,small
4,20.29,14.34,135.10,1297.0,0.10030,0,large


In [2]:
df["mean area"]

0      1001.0
1      1326.0
2      1203.0
3       386.1
4      1297.0
        ...  
564    1479.0
565    1261.0
566     858.1
567    1265.0
568     181.0
Name: mean area, Length: 569, dtype: float64

In [3]:
def run_checks(title, checks):
    """checks: список (описание, callable->bool). Печатает PASS/FAIL и падает, если не всё пройдено."""
    print(title)
    ok_count = 0
    for desc, cond in checks:
        try:
            ok = bool(cond())
        except Exception as e:
            ok, desc = False, f"{desc}  [ошибка: {type(e).__name__}: {e}]"
        print(f"  [{'PASS' if ok else 'FAIL'}] {desc}")
        ok_count += ok
    print(f"  -> {ok_count}/{len(checks)} проверок пройдено")
    assert ok_count == len(checks), "Не все проверки пройдены — доработайте функцию."

### Кирпичик 1 — стратифицированный train/test split

**`train_test_split(X, y, ...)`** — случайно делит выборку на обучающую и тестовую части.
`test_size` задаёт долю теста, `stratify=y` сохраняет доли классов в обеих частях,
`random_state` фиксирует разбиение (воспроизводимость).
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

**Задание.** Реализуйте `make_split(X, y)` с `test_size=0.25`, `stratify=y`, `random_state=SEED`.
Верните `X_train, X_test, y_train, y_test`. Сплит делается **до** любой предобработки.

In [4]:
def make_split(X, y):
    # TODO: верните стратифицированное разбиение (test_size=0.25, stratify=y, random_state=SEED)
    #       в порядке: X_train, X_test, y_train, y_test
    test_size=0.25
    N=len(y)
    N1 = np.sum(y>0.5)
    N0=N-N1

    indices0 = np.where(y==0)[0]
    indices1 = np.where(y==1)[0]

    bin_str0 = np.array(random.choices([0, 1], weights=[1-test_size, test_size], k=len(indices0)))
    inidices0_train = indices0[np.where(bin_str0==0)]
    inidices0_test = indices0[np.where(bin_str0==1)]

    bin_str1 = np.array(random.choices([0, 1], weights=[1-test_size, test_size], k=len(indices1)))
    inidices1_train = indices1[np.where(bin_str1==0)]
    inidices1_test = indices1[np.where(bin_str1==1)]

    indices_train = np.concat((inidices0_train, inidices1_train))
    X_train = X.iloc[indices_train]
    y_train = y.iloc[indices_train]
    indices_test = np.concat((inidices0_test, inidices1_test))
    X_test = X.iloc[indices_test]
    y_test = y.iloc[indices_test]

    return X_train, X_test, y_train, y_test

In [5]:
Xtr, Xte, ytr, yte = make_split(X, y)
_rXtr, _rXte, _rytr, _ryte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

run_checks("Кирпичик 1 — стратифицированный split", [
    ("Размер теста = 25%",                     lambda: len(Xte) == len(_rXte)),
    ("Train и test не пересекаются",           lambda: len(set(Xtr.index) & set(Xte.index)) == 0),
    ("Покрыты все объекты",                    lambda: len(Xtr) + len(Xte) == len(X)),
    ("Индексы теста = стратиф. эталон",        lambda: set(Xte.index) == set(_rXte.index)),
    ("Доля классов сохранена (стратификация)", lambda: abs(ytr.mean() - y.mean()) < 0.02),
])

NameError: name 'random' is not defined

### Кирпичик 2 — заполнение пропусков без утечки

**`Series.median()`** — медиана столбца, пропуски (`NaN`) игнорируются.
[Документация →](https://pandas.pydata.org/docs/reference/api/pandas.Series.median.html)
**`Series.fillna(value)`** — заменяет `NaN` на переданное значение.
[Документация →](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html)
*(В реальном пайплайне то же делает [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) внутри `Pipeline`.)*

**Задание.** Реализуйте `impute_no_leak(train_col, test_col)`: посчитайте медиану **только по train**
и заполните ею пропуски и в train, и в test. Верните `(train_filled, test_filled)`.
Медиана по всем данным — это утечка.

In [6]:
def impute_no_leak(train_col, test_col):
    median = train_col.median()

    train_filled = train_col.fillna(median)
    test_filled = test_col.fillna(median)
    # TODO: посчитайте медиану ТОЛЬКО по train_col и заполните ею пропуски и в train, и в test
    return train_filled, test_filled

In [7]:
_tr, _te = X_train["mean smoothness"], X_test["mean smoothness"]
_trf, _tef = impute_no_leak(_tr, _te)
_train_med = _tr.median()
_full_med = pd.concat([_tr, _te]).median()

run_checks("Кирпичик 2 — импутация без утечки", [
    ("В train не осталось пропусков",              lambda: int(_trf.isna().sum()) == 0),
    ("В test не осталось пропусков",               lambda: int(_tef.isna().sum()) == 0),
    ("Пропуски заполнены медианой TRAIN",          lambda: np.allclose(_tef[_te.isna()].values, _train_med)),
    ("Медиана train != медианы всех данных (есть чем отличить утечку)",
        lambda: abs(_train_med - _full_med) > 1e-9),
    ("Не использована медиана всех данных (нет утечки)",
        lambda: not np.allclose(_tef[_te.isna()].values, _full_med)),
])

Кирпичик 2 — импутация без утечки
  [PASS] В train не осталось пропусков
  [PASS] В test не осталось пропусков
  [PASS] Пропуски заполнены медианой TRAIN
  [PASS] Медиана train != медианы всех данных (есть чем отличить утечку)
  [PASS] Не использована медиана всех данных (нет утечки)
  -> 5/5 проверок пройдено


### Кирпичик 3 — One-Hot кодирование категорий

**`OneHotEncoder`** — превращает категориальный столбец в набор бинарных столбцов-индикаторов
(по одному на категорию). Обучается на train (`fit_transform`), применяется к test (`transform`);
`handle_unknown="ignore"` безопасно обрабатывает не встречавшиеся категории.
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)

**Задание.** Реализуйте `one_hot(train_cat, test_cat)`: обучите энкодер на train и примените к train и test.
Верните `(enc_train, enc_test)` как numpy-массивы.

In [8]:
def one_hot(train_cat, test_cat):
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).set_output(transform='pandas')
    enc_train = ohe.fit_transform(train_cat.to_frame())
    enc_test = ohe.transform(test_cat.to_frame())
    # TODO: обучите OneHotEncoder на train_cat, примените к train_cat и test_cat
    #       верните два numpy-массива (для train и для test)
    return enc_train, enc_test

In [9]:
_a, _b = one_hot(X_train[CAT_COLS[0]], X_test[CAT_COLS[0]])
_ref = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
_ra = _ref.fit_transform(X_train[[CAT_COLS[0]]])
_rb = _ref.transform(X_test[[CAT_COLS[0]]])

run_checks("Кирпичик 3 — One-Hot кодирование", [
    ("Число колонок = число категорий train", lambda: np.asarray(_a).shape[1] == len(_ref.categories_[0])),
    ("train закодирован верно",               lambda: np.allclose(np.asarray(_a), _ra)),
    ("test закодирован тем же энкодером",      lambda: np.allclose(np.asarray(_b), _rb)),
    ("В каждой строке train ровно один бит",   lambda: np.allclose(np.asarray(_a).sum(axis=1), 1)),
])

Кирпичик 3 — One-Hot кодирование
  [PASS] Число колонок = число категорий train
  [PASS] train закодирован верно
  [PASS] test закодирован тем же энкодером
  [PASS] В каждой строке train ровно один бит
  -> 4/4 проверок пройдено


### Кирпичик 4 — метрики из ошибок (руками)

По матрице ошибок (`TP/FP/FN/TN`) считаются:
$\text{precision}=\frac{TP}{TP+FP}$, $\text{recall}=\frac{TP}{TP+FN}$,
$F_1$ — гармоническое среднее precision и recall, $\text{accuracy}=\frac{TP+TN}{\text{всего}}$.
Эталонные реализации:
[precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html) ·
[recall](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html) ·
[f1](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html) ·
[accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html) ·
[confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

**Задание.** Реализуйте `manual_metrics(y_true, y_pred)` (класс 1 — положительный) и верните словарь
с ключами `"precision"`, `"recall"`, `"f1"`, `"accuracy"`.

In [10]:
def manual_metrics(y_true, y_pred):
    TP_arr = np.where((y_true==1) & (y_pred==1))[0]
    TP = len(TP_arr)

    TN_arr = np.where((y_true==0) & (y_pred==0))[0]
    TN = len(TN_arr)

    FP_arr = np.where((y_true==0) & (y_pred==1))[0]
    FP = len(FP_arr)

    FN_arr = np.where((y_true==1) & (y_pred==0))[0]
    FN = len(FN_arr)
    precision = TP/(TP+FP)
    recall = TP/(TP+FN)
    f1 = 2*precision*recall/(precision + recall)
    accuracy = (TP+TN)/(TP+TN+FP+FN)
    # TODO: посчитайте TP, FP, FN, TN и по ним precision, recall, f1, accuracy
    return {'precision':TP/(TP+FP),
             'recall': TP/(TP+FN),
             'f1': f1,
             'accuracy': accuracy
    }   

In [11]:
_yt = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
_yp = np.array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0])
_m = manual_metrics(_yt, _yp)

run_checks("Кирпичик 4 — метрики вручную", [
    ("precision верна", lambda: abs(_m["precision"] - precision_score(_yt, _yp)) < 1e-9),
    ("recall верна",    lambda: abs(_m["recall"]    - recall_score(_yt, _yp))    < 1e-9),
    ("f1 верна",        lambda: abs(_m["f1"]        - f1_score(_yt, _yp))        < 1e-9),
    ("accuracy верна",  lambda: abs(_m["accuracy"]  - accuracy_score(_yt, _yp))  < 1e-9),
])

Кирпичик 4 — метрики вручную
  [PASS] precision верна
  [PASS] recall верна
  [PASS] f1 верна
  [PASS] accuracy верна
  -> 4/4 проверок пройдено


### Кирпичик 5 — почему не accuracy при дисбалансе

**`roc_auc_score(y_true, proba)`** — площадь под ROC-кривой. Интерпретация: вероятность, что
случайный положительный объект получит больший скор, чем случайный отрицательный. Принимает
вероятности/скор, а не метки. AUC = 0.5 — случайная модель.
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)

**Задание.** Реализуйте `auc_and_dummy(y_true, proba)`: верните `(roc_auc, dummy_accuracy)`, где
`dummy_accuracy` — доля мажоритарного класса (accuracy тривиального «всегда мажоритарный класс»).

In [12]:
def auc_and_dummy(y_true, proba):
    roc = roc_auc_score(y_true, proba)
    frac_of1= np.mean(y_true)
    dummy_accuracy = max(frac_of1, 1-frac_of1)
    # TODO: roc_auc по proba; dummy_accuracy = доля мажоритарного класса
    return roc, dummy_accuracy

In [13]:
_auc, _dummy 

NameError: name '_auc' is not defined

In [14]:
_yt = np.array([0] * 90 + [1] * 10)
_proba = np.concatenate([np.linspace(0.10, 0.60, 90), np.linspace(0.40, 0.95, 10)])
_auc, _dummy = auc_and_dummy(_yt, _proba)

run_checks("Кирпичик 5 — ROC-AUC vs accuracy при дисбалансе", [
    ("ROC-AUC верна",                          lambda: abs(_auc - roc_auc_score(_yt, _proba)) < 1e-9),
    ("Dummy-accuracy = доля мажор. класса",    lambda: abs(_dummy - 0.90) < 1e-9),
    ("Урок: dummy-accuracy >= 0.9, но модель бесполезна", lambda: _dummy >= 0.9),
])

Кирпичик 5 — ROC-AUC vs accuracy при дисбалансе
  [PASS] ROC-AUC верна
  [PASS] Dummy-accuracy = доля мажор. класса
  [PASS] Урок: dummy-accuracy >= 0.9, но модель бесполезна
  -> 3/3 проверок пройдено


### Кирпичик 6 — честная кросс-валидация

**`cross_val_score(estimator, X, y, cv, scoring)`** — обучает и оценивает модель на нескольких
фолдах, возвращает массив оценок (по одной на фолд).
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
**`StratifiedKFold`** — разбиение на фолды с сохранением долей классов (объект `CV` уже готов).
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)

**Задание.** Реализуйте `cv_auc(estimator, X, y)`: оцените `estimator` по метрике `roc_auc`
на 5 фолдах (`cv=CV`). Верните массив из 5 значений.

In [29]:
def cv_auc(estimator, X, y):
    # TODO: оцените estimator стратифицированной 5-fold кросс-валидацией (объект CV)
    #       по метрике roc_auc; верните массив из 5 значений
    skf = StratifiedKFold(n_splits=5)

    roc_auc_scores = []
    for train_index, test_index in CV.split(X, y):
        X_train = Xn.iloc[train_index]
        y_train = y.iloc[train_index]

        X_test = Xn.iloc[test_index]
        y_test = y.iloc[test_index]
        estimator.fit(X_train, y_train)
        y_proba = estimator.predict_proba(X_test)[:,1]
        score = roc_auc_score(y_test, y_proba)
        roc_auc_scores.append(score)
    return roc_auc_scores

In [30]:
_scores = cv_auc(make_estimator(), Xn, y)
_ref = cross_val_score(make_estimator(), Xn, y, cv=CV, scoring="roc_auc")

run_checks("Кирпичик 6 — стратифицированная кросс-валидация", [
    ("5 значений (5 фолдов)",                    lambda: len(_scores) == 5),
    ("Совпадает с эталоном (тот же seed/метрика)", lambda: np.allclose(np.sort(_scores), np.sort(_ref))),
    ("Средний ROC-AUC разумный (> 0.9)",         lambda: float(np.mean(_scores)) > 0.9),
])

Кирпичик 6 — стратифицированная кросс-валидация
  [PASS] 5 значений (5 фолдов)
  [PASS] Совпадает с эталоном (тот же seed/метрика)
  [PASS] Средний ROC-AUC разумный (> 0.9)
  -> 3/3 проверок пройдено


### Кирпичик 7 — подбор гиперпараметра (GridSearchCV)

**`GridSearchCV(estimator, param_grid, cv, scoring)`** — перебирает все комбинации гиперпараметров
из сетки с кросс-валидацией; после `fit` лучшие лежат в `best_params_`, лучшая модель — в `best_estimator_`.
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

**Задание.** Реализуйте `best_C(X, y)`: переберите сетку `PARAM_GRID` (базовый оценщик `make_estimator()`,
`cv=CV`, `scoring="roc_auc"`). Верните лучшее значение `C` из `best_params_["logisticregression__C"]`.

In [32]:
def best_C(X, y):
    gs= GridSearchCV(make_estimator(), PARAM_GRID, cv=CV, scoring="roc_auc")
    gs.fit(X,y)
    # TODO: переберите PARAM_GRID через GridSearchCV (оценщик make_estimator(), cv=CV,
    #       метрика roc_auc); верните лучшее C из best_params_
    return gs.best_params_["logisticregression__C"]


In [33]:
_bc = best_C(Xn, y)
_ref_c = GridSearchCV(make_estimator(), PARAM_GRID, cv=CV, scoring="roc_auc").fit(Xn, y).best_params_["logisticregression__C"]

run_checks("Кирпичик 7 — GridSearchCV подбор C", [
    ("C из заданной сетки",   lambda: _bc in C_GRID),
    ("C совпал с эталоном",   lambda: _bc == _ref_c),
])

Кирпичик 7 — GridSearchCV подбор C
  [PASS] C из заданной сетки
  [PASS] C совпал с эталоном
  -> 2/2 проверок пройдено


### Кирпичик 8 — важность признаков (топ-3)

**`DecisionTreeClassifier`** — решающее дерево. После `fit` атрибут `feature_importances_`
показывает вклад каждого признака (сумма = 1); чем больше, тем важнее признак для модели.
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)

**Задание.** Реализуйте `top3_features(X, y)`: обучите `DecisionTreeClassifier(random_state=SEED)`
и верните список из 3 названий признаков с наибольшей важностью (по убыванию).

In [43]:
def top3_features(X, y):
    # TODO: обучите DecisionTreeClassifier(random_state=SEED); возьмите feature_importances_
    #       верните 3 названия столбцов с наибольшей важностью (по убыванию)
    dec_tree = DecisionTreeClassifier(random_state=SEED).fit(X,y)
    feat_imp = dec_tree.feature_importances_
    top3_indices = np.argsort(feat_imp)[::-1][:3]
    return list(X.columns[top3_indices])

In [44]:
_top = top3_features(Xn, y)
_t = DecisionTreeClassifier(random_state=SEED).fit(Xn, y)
_ref_top = list(np.array(Xn.columns)[np.argsort(_t.feature_importances_)[::-1]][:3])

run_checks("Кирпичик 8 — важность признаков (топ-3)", [
    ("Вернулось 3 признака",         lambda: len(_top) == 3),
    ("Это реальные столбцы",         lambda: all(f in list(Xn.columns) for f in _top)),
    ("Топ-3 совпал с эталоном",      lambda: list(_top) == _ref_top),
])

Кирпичик 8 — важность признаков (топ-3)
  [PASS] Вернулось 3 признака
  [PASS] Это реальные столбцы
  [PASS] Топ-3 совпал с эталоном
  -> 3/3 проверок пройдено
